In [7]:
import time
from time import sleep, monotonic
import datetime
import numpy as np
import matplotlib.pyplot as plt
import sys
import pyvisa
import qcodes as qc
from qcodes.dataset import Measurement
from qcodes.dataset import do0d
from qcodes.dataset.experiment_container import new_experiment, load_experiment_by_name
from qcodes.dataset.plotting import plot_by_id
from qcodes.dataset.data_set import load_by_id, load_by_counter
from qcodes import initialise_or_create_database_at, new_data_set, new_experiment
from qcodes.station import Station
initialise_or_create_database_at("./2026-05-25-mso5_count_test.db")
import snspd
params = snspd.snspd('snspd11.yaml')

# Set up experiment
exp_name = 'mso5'
sample_name = '00'

try:
    exp = qc.load_experiment_by_name(exp_name, sample=sample_name)
    print('Experiment loaded. Last ID no:', exp.last_counter)
except ValueError:
    exp = new_experiment(exp_name, sample_name)
    print('Started new experiment')

Experiment loaded. Last ID no: 0


In [6]:
station = Station(config_file="friesland.yaml")
dmm = station.load_instrument("dmm", revive_instance=True)
yoko = station.load_instrument("yoko", revive_instance=True)
# laser = station.load_instrument("laser", revive_instance=True)
MS = station.load_instrument("osc", revive_instance=True)
# pm100d = station.load_instrument("pm100d", revive_instance=True) 
# pms120 = station.load_instrument("pms120", revive_instance=True)
# tc = station.load_instrument("fridge", revive_instance=True)
# p_att = station.load_instrument("dmm_keithley", revive_instance=True) # excluding from snapshot because none of the parameters work anyway

2026-05-25 14:20:53,060 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ D:\SNSPD\SNSPD2\MSO5.py:5: QCoDeSDeprecationWarning: The `qcodes.instrument.base` module is deprecated. Please consult the api documentation at https://microsoft.github.io/Qcodes/api/index.html for alternatives.
  from qcodes.instrument.base import InstrumentBase



In [9]:
params.MSO5_set_standard_counts(device=params.device_line_1, MS=MS) # vertical scale will pertain to device_line_1 parameters 

In [12]:
meas = Measurement()
meas.register_parameter(dmm.volt)
meas.register_parameter(yoko.current)
meas.register_custom_parameter("wavelength", label="m")
meas.register_custom_parameter("v_attenuator", label="V")
meas.register_custom_parameter("threshold1", label="V")
meas.register_custom_parameter("threshold2", label="V")
meas.register_custom_parameter("total_counts1", label="counts")
meas.register_custom_parameter("total_counts2", label="counts")
meas.register_custom_parameter("counts1")
meas.register_custom_parameter("counts2")
meas.register_custom_parameter("trace_time", label="s")
meas.register_custom_parameter("meas_time", label="s")
meas.register_custom_parameter("interval", label="s")
meas.register_custom_parameter("CR1", label="cps", setpoints=(yoko.current,))
meas.register_custom_parameter("CR2", label="cps", setpoints=(yoko.current,))
meas.register_custom_parameter("n_captures")
meas.register_custom_parameter("power90", label="W")
meas.register_custom_parameter('v_scale', label='V')


with meas.run() as datasaver:
    print(datasaver.run_id)
    
    threshold1 = 250e-3
    threshold2 = 100e-3
    n_captures = 10 
    interval = 1 
    osc = MS
    
    # Set thresholds on oscilloscope   
    osc.write(f'SEARCH:SEARCH1:TRIGger:A:EDGE:THReshold {threshold1}')
    osc.write(f'SEARCH:SEARCH2:TRIGger:A:EDGE:THReshold {threshold2}')
    
    # # Set corresponding vertical scaling 
    # osc.channels[0].vertical_scale(v_scale)
    
    print(osc.channels[0].vertical_scale( ))
    
    time.sleep(5)
    
    if osc.channels[0].clipping(): 
        raise Exception('Error: Clipping')
    
    time.sleep(5)
    
    # Run count 
    osc.write("SEARCH:SEARCH1:STATE 0")
    osc.write("SEARCH:SEARCH1:STATE 1")
    osc.write("SEARCH:SEARCH2:STATE 0")
    osc.write("SEARCH:SEARCH2:STATE 1")
    
    start = time.perf_counter()
    print(f'This acquisition will take {n_captures*interval}s')
    print(datetime.datetime.now().hour,  datetime.datetime.now().minute)
    
    counts1= []
    counts2= []
    
    
    for i in range(n_captures):
        time.sleep(interval)
    
        # Extract counts 
        count1 = int(osc.ask("SEARCH:SEARCH1:TOTal?"))
        count2 = int(osc.ask("SEARCH:SEARCH2:TOTal?"))
    
        counts1.append(count1)
        counts2.append(count2)
    
        
    # calculate total counts 
    total_counts1 = sum(counts1)
    total_counts2 = sum(counts2)
    
    # Extract the amount of time in one trace 
    h_time = osc.horizontal_scale()*osc.horizontal_divisions()
    
    # total time in measurement 
    meas_time = n_captures*h_time
    
    # dark count rate calculation
    CR1 = total_counts1/meas_time
    CR2 = total_counts2/meas_time


    # Save data 
    datasaver.add_result((yoko.current, yoko.current()),
                        (dmm.volt, dmm.volt()),
                        ("threshold1",  float(osc.ask(f'SEARCH:SEARCH1:TRIGger:A:EDGE:THReshold?'))), 
                        ("threshold2",  float(osc.ask(f'SEARCH:SEARCH2:TRIGger:A:EDGE:THReshold?'))), 
                        ("total_counts1", total_counts1), 
                        ("total_counts2", total_counts2), 
                        ("counts1", counts1), 
                        ("counts2", counts2), 
                        ("meas_time", meas_time), 
                        ("interval", interval), 
                        ("n_captures", n_captures),
                        ("CR1", CR1), 
                        ("CR2", CR2),
                        ("v_scale", float(osc.channels[0].vertical_scale())))

0.15
This acquisition will take 10s
14 26
